# 04. 이벤트 기준선과 충격·회복 계산 (event baseline)

**이 노트북이 하는 일**

특보 이벤트마다 **"특보가 없었다면 이랬을 이용량(기준선)"**을 예측하고, 실제 이용량이 기준선의 95% 구간 아래로 얼마나 내려갔는지(충격), 해제 후 얼마 만에 돌아왔는지(회복)를 역별로 계산합니다 (RQ1·2, 피드백 2).

**기준선 모형**

- SARIMA (1,0,1)(1,1,1,7), 외생변수는 계단 더미와 `summer_vacation`만 (날씨·특보 변수 없음 = 03번의 N 모형). 9개 역 각각 2014~2023으로 한 번 적합하고 계수는 고정합니다.
- y(= log 이용량)는 `MASK_COLS`가 1인 날에 더해 **`any_alert`=1인 날과 해제 후 7일**도 NaN으로 가립니다.
- 이벤트마다 **시작 전날까지의 자료를 붙이고(`apply`, `refit=False`) 지속일수 + 14일을 예측**해 기준선과 95% 구간을 얻습니다.

**판정 규칙**

- 편차 `d = (실제 − 기준선) / 기준선`, 최대 감소율 = 이벤트 기간 중 `−d`의 최댓값, 누적 손실 = (기준선 − 실제)의 합.
- **충격**: 이벤트 기간 중 실제가 구간 **아래**인 날이 **k일 이상** (이벤트가 k일보다 짧으면 지속일수만큼).
- **회복일**: 해제 후 실제가 구간 **안**에 **m일 연속**(`RECOVERY_M` = **2**, 고정) 머문 첫날. 회복 지연 = 회복일 − 해제일. 해제 후 14일 안에 못 돌아오면 **중도절단**. 회복 확인 창은 **다음 특보가 시작되거나 가림일(공휴일 등)이 나오기 전날까지**만 봅니다 (그 안에 못 돌아오면 중도절단. 다음 특보가 해제 후 8~14일째 시작해도 그 전날까지만 확인합니다).
- **기준선 근거 규칙**: 시작 전까지 관측된 날(y가 가려지지 않은 날)이 `EVENT_MIN_HISTORY`(28)일 미만인 이벤트는 기준선을 믿을 수 없어 제외합니다 (2014년 초 이벤트).
- **placebo**: 특보도 가림일도 없는 가짜 시작일 300개에 같은 절차를 적용해, 충격 오판정이 5% 이하인 가장 작은 **k(1~3)**를 고릅니다. **m은 placebo로 고르지 않고 2로 고정**하며, m=1, m=3 결과는 민감도 표로 같이 봅니다.
- **회복 결과에는 회복 판정이 가능했던 이벤트 수(중도절단 제외)를 꼭 같이 표시**합니다.

**결과는 세 가지**: 단일(주 분석) / 복합 / 민감도 (02번 노트북의 기준). 대상은 **train+validation(2014~2024)의 깨끗한 이벤트**이고 대설은 참고용입니다. **2025년 행은 읽자마자 버리며 쓰지 않습니다.**

**입력 파일**

- `data/processed/model_input_9stations.csv`, `results/events.csv` (02번 출력)

**출력 파일**

- `results/event_effects.csv` (이벤트 × 역별 충격·회복), `event_by_station_maxdrop.csv`, `event_by_station_recovery.csv` (이벤트 × 역 나란히 보기), `event_summary_by_type.csv` (유형 × 종류 요약), `event_daily_paths.csv` (일별 실제·기준선·구간), `event_placebo.csv`, `event_placebo_rates.csv`, `event_baseline_fits.csv`, `event_long_events.csv`


**설정 불러오기**: Colab이면 저장소를 clone해 경로를 맞추고, `00_settings.ipynb`의 설정을 불러옵니다 (로컬이면 clone은 건너뜀).

In [1]:
# Colab에서 실행하면 저장소를 clone하고 notebooks/로 이동합니다 (로컬이면 건너뜁니다).
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/TimeSeries'):
        subprocess.run(['git', 'clone', 'https://github.com/Eui-seung/TimeSeries.git', '/content/TimeSeries'], check=True)
    os.chdir('/content/TimeSeries/notebooks')

# 설정(경로, 역, 열 이름, 기간, 차수 등)을 불러옵니다.
%run ./00_settings.ipynb

필요한 패키지를 불러옵니다. 역별 적합은 `joblib`으로 병렬 처리합니다.

In [2]:
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import wilcoxon
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

**입력 읽기**: 모델 입력 표는 **2024-12-31까지만** 읽고(2025년 행은 버림), 02번에서 만든 이벤트 표를 읽습니다.

In [3]:
mi = pd.read_csv(PROC / 'model_input_9stations.csv', parse_dates=['date'])
mi = mi[mi.date <= VALID_END]                                   # 2025년 행은 여기서 버림
ev_all = pd.read_csv(RESULTS / 'events.csv', parse_dates=['시작일', '종료일', '다음특보시작일'])
print('모델 입력', mi.shape, mi.date.min().date(), '~', mi.date.max().date(), '| 이벤트 표', ev_all.shape)

모델 입력 (36162, 44) 2014-01-01 ~ 2024-12-31 | 이벤트 표 (219, 28)


**날짜별 달력 표**: 가림일, 특보일, 계단 더미 시행일 ±30일, 그리고 **y가 관측되는 날**(가림일과 "특보일+해제 후 7일"을 뺀 날)을 만듭니다. 시작 전까지 관측된 날 수(`obs_before`)는 기준선의 근거 자료량입니다.

In [4]:
cal = mi[mi.station == STATIONS[0]].set_index('date').sort_index()
step_dates = {c: mi.loc[mi[c] == 1, 'date'].min() for c in STEP_COLS}
step_dates.update({c: mi.loc[(mi.station == st) & (mi[c] == 1), 'date'].min() for st, c in STATION_STEP.items()})
near_step = pd.Series(False, index=cal.index)
for d in step_dates.values():
    near_step |= (cal.index >= d - pd.Timedelta(days=EVENT_STEP_PM_DAYS)) & (cal.index <= d + pd.Timedelta(days=EVENT_STEP_PM_DAYS))
mask_day = cal[MASK_COLS].sum(axis=1) > 0                        # 가림일(공휴일, 명절, 코로나 등)
alert_day = cal.any_alert.fillna(0) > 0                          # 특보가 켜진 날
alert_mask = alert_day.copy()                                    # 특보일과 해제 후 7일 (y를 가리는 날)
for k in range(1, ALERT_MASK_AFTER_DAYS + 1):
    alert_mask |= alert_day.shift(k, fill_value=False)
obs_day = ~(mask_day | alert_mask)                               # y가 관측되는 날 (모든 역이 같음)
obs_before = obs_day.cumsum().shift(1, fill_value=0)             # 그날 전까지 관측된 날 수
print('가림일', int(mask_day.sum()), '일 | 특보일', int(alert_day.sum()), '일 | 특보일+해제 후 7일', int(alert_mask.sum()), '일 | 계단 더미 ±30일', int(near_step.sum()), '일 | y 관측일', int(obs_day.sum()), '일')

가림일 1123 일 | 특보일 635 일 | 특보일+해제 후 7일 1458 일 | 계단 더미 ±30일 244 일 | y 관측일 1854 일


**분석 이벤트 고르기**: 02번 이벤트 표에서 **train+validation의 깨끗한 이벤트**를 고르고, **시작 전 관측일이 `EVENT_MIN_HISTORY`(28)일 미만인 이벤트는 제외**합니다. 단일(주 분석)과 복합을 합친 것이 이번 분석 대상이고, 민감도는 단일의 부분집합입니다.

In [5]:
ev0 = ev_all[ev_all.깨끗함 & (ev_all.기간 != 'test')].copy()
ev0['obs_before'] = [int(obs_before.loc[d]) for d in ev0.시작일]
short = ev0[ev0.obs_before < EVENT_MIN_HISTORY]
ev = ev0[ev0.obs_before >= EVENT_MIN_HISTORY].reset_index(drop=True)
print('깨끗한 이벤트(train+val)', len(ev0), '개 중 기준선 근거 부족으로 제외:', len(short), '개 →', short[['이벤트번호', '시작일', 'obs_before']].values.tolist())
print('분석 대상 이벤트', len(ev), '개: 단일', int((ev.분류 == '단일').sum()), '/ 복합', int((ev.분류 == '복합').sum()), '/ 민감도(단일의 부분집합)', int(ev.민감도단일.sum()))
print(ev.groupby(['분류', '종류']).size().to_string())
print('long_event(14일 초과):', int(ev.long_event.sum()), '개')

깨끗한 이벤트(train+val) 98 개 중 기준선 근거 부족으로 제외: 2 개 → [['단일-대설-01', Timestamp('2014-01-20 00:00:00'), 2], ['단일-대설-02', Timestamp('2014-02-08 00:00:00'), 8]]
분석 대상 이벤트 96 개: 단일 93 / 복합 3 / 민감도(단일의 부분집합) 50
분류  종류   
단일  대설        8
    폭염       27
    한파       19
    호우       39
복합  폭염+호우     3
long_event(14일 초과): 0 개


**placebo 가짜 이벤트 뽑기**: 실제 분석 이벤트의 지속일수 분포에서 지속일수를 뽑고(300개), 그 길이의 확인 창(시작 ~ 종료+7일)에 **특보·가림일·계단 더미 ±30일이 없는** 시작일을 무작위로 고릅니다 (깨끗한 이벤트와 같은 조건). 시작일은 시작 전 관측일이 `EVENT_MIN_HISTORY`(28)일 이상이고, 예측이 2024-12-31 안에 끝나야 합니다.

In [6]:
rng = np.random.default_rng(PLACEBO_SEED)
ok_day = ~(mask_day | alert_day | near_step)                      # 하루가 "깨끗한가"
first_ok, last_end = pd.Timestamp('2014-02-01'), VALID_END
durs = ev.지속일수.to_numpy()
fakes, used = [], set()
while len(fakes) < PLACEBO_N:
    D = int(rng.choice(durs))
    w = D + ALERT_MASK_AFTER_DAYS
    win_ok = ok_day.astype(int).rolling(w).sum().shift(-(w - 1)) == w         # t에서 시작하는 창이 모두 깨끗한가
    cand = win_ok[win_ok & (win_ok.index >= first_ok) & (obs_before.reindex(win_ok.index) >= EVENT_MIN_HISTORY) & (win_ok.index + pd.Timedelta(days=D + EVENT_POST_DAYS - 1) <= last_end)].index
    cand = [c for c in cand if c not in used]
    if not cand:
        continue
    t = cand[int(rng.integers(len(cand)))]
    used.add(t)
    fakes.append({'id': f'P{len(fakes) + 1:03d}', 'start': t, 'end': t + pd.Timedelta(days=D - 1), 'D': D})
fk = pd.DataFrame(fakes)
print('가짜 이벤트', len(fk), '개 | 지속일수 평균', round(fk.D.mean(), 1), '| 시작 연도 분포:', fk.start.dt.year.value_counts().sort_index().to_dict())

가짜 이벤트 300 개 | 지속일수 평균 2.3 | 시작 연도 분포: {2014: 48, 2015: 48, 2016: 25, 2017: 30, 2018: 34, 2019: 35, 2020: 3, 2022: 23, 2023: 28, 2024: 26}


**이벤트 목록 만들기**: 진짜 이벤트와 가짜 이벤트를 같은 형식(`id`, `start`, `end`, `H`=예측 기간=지속일수+14일)으로 모읍니다.

In [7]:
jobs = [{'id': r.이벤트번호, 'start': r.시작일, 'end': r.종료일, 'H': int(r.지속일수) + EVENT_POST_DAYS} for r in ev.itertuples()]
jobs += [{'id': r.id, 'start': r.start, 'end': r.end, 'H': int(r.D) + EVENT_POST_DAYS} for r in fk.itertuples()]
print('예측할 이벤트', len(jobs), '개 (진짜', len(ev), '+ 가짜', len(fk), ') × 역', len(STATIONS), '개')

예측할 이벤트 396 개 (진짜 96 + 가짜 300 ) × 역 9 개


**`station_task`**: 한 역에서 ① y(log 이용량)에 가림(`MASK_COLS` + 특보일 + 해제 후 7일)을 적용하고 ② 계단 더미·`summer_vacation`을 외생변수로 2014~2023에 적합(학습 구간에 값이 전부 0인 더미는 뺌, 미수렴이면 이어서 다시 적합)한 뒤 ③ 계수를 고정한 채 전체 기간에 적용(`apply`)해서 ④ 이벤트마다 시작일부터 `H`일을 **동적 예측**(시작 전날까지의 자료만 사용)합니다. 반환: 일별 실제·기준선·95% 구간 표와 적합 요약.

In [8]:
def station_task(st, jobs):
    warnings.filterwarnings('ignore')
    d = pd.read_csv(PROC / 'model_input_9stations.csv', parse_dates=['date'])
    d = d[(d.station == st) & (d.date <= VALID_END)]                        # 2025년 행은 읽자마자 버림
    s = d.set_index('date').sort_index().asfreq('D')
    y = np.log(s.total)
    y[s[MASK_COLS].sum(axis=1) > 0] = np.nan                                # 기본 가림
    a = s.any_alert.fillna(0)
    am = a > 0
    for k in range(1, ALERT_MASK_AFTER_DAYS + 1):
        am |= (a.shift(k).fillna(0) > 0)                                    # 특보일과 해제 후 7일
    y[am] = np.nan
    X = s[STEP_COLS + SEASON_COLS + ([STATION_STEP[st]] if st in STATION_STEP else [])].astype(float)
    Xtr = X[:TRAIN_END]
    drop = [c for c in X.columns if (Xtr[c] == 0).all()]                    # 학습 중 전부 0인 더미(fare_2025, climate_card 등)
    X = X.drop(columns=drop)
    model = SARIMAX(y[:TRAIN_END], exog=X[:TRAIN_END], order=ORDER, seasonal_order=SEASONAL_ORDER)
    res = model.fit(disp=False, method=FIT_METHOD, maxiter=FIT_MAXITER)
    if not res.mle_retvals.get('converged'):                                # 미수렴이면 이어서 다시
        res = model.fit(start_params=res.params.values, disp=False, method=FIT_METHOD, maxiter=5000)
    full = res.apply(y, exog=X, refit=False)                                # 계수 고정, 2014~2024 전체에 적용
    mday = (s[MASK_COLS].sum(axis=1) > 0)
    rows = []
    for j in jobs:
        e_last = min(j['start'] + pd.Timedelta(days=j['H'] - 1), VALID_END)  # 2024-12-31을 넘지 않게 자름
        H = (e_last - j['start']).days + 1
        p = full.get_prediction(start=j['start'], end=e_last, dynamic=0)     # 시작 전날까지의 자료만 쓰는 동적 예측
        ci = p.conf_int(alpha=BAND_ALPHA)
        g = pd.DataFrame({'actual': s.total.loc[j['start']:e_last], 'base': np.exp(p.predicted_mean), 'lower': np.exp(ci.iloc[:, 0]), 'upper': np.exp(ci.iloc[:, 1]),
                          'mask_day': mday.loc[j['start']:e_last], 'alert_day': (a > 0).loc[j['start']:e_last]})
        g.insert(0, 'id', j['id']); g.insert(1, '역', st); g.insert(2, 'end', j['end'])
        g['prev_obs60'] = int(y.loc[j['start'] - pd.Timedelta(days=60):j['start'] - pd.Timedelta(days=1)].notna().sum())
        rows.append(g)
    fit = {'역': st, '유형': s.group.iloc[0], 'AIC': res.aic, '수렴': bool(res.mle_retvals.get('converged')), '학습 y 관측일수': int(y[:TRAIN_END].notna().sum()),
           '학습 일수': len(y[:TRAIN_END]), '뺀 더미': ','.join(drop)}
    return pd.concat(rows).reset_index().rename(columns={'index': 'date'}), fit

**9개 역 병렬 실행**: 역마다 한 번 적합하고 진짜·가짜 이벤트 전부를 예측합니다 (역당 약 10~30초).

In [9]:
out = Parallel(n_jobs=WORKERS)(delayed(station_task)(st, jobs) for st in STATIONS)
paths = pd.concat([o[0] for o in out], ignore_index=True)
fits = pd.DataFrame([o[1] for o in out])
fits.to_csv(RESULTS / 'event_baseline_fits.csv', index=False, encoding='utf-8-sig')
print('일별 예측 표', paths.shape)
fits

일별 예측 표 (58239, 11)


,역,유형,AIC,수렴,학습 y 관측일수,학습 일수,뺀 더미
0,시청,Employment,-3597.850103,True,1656,3652,"fare_2025,climate_card"
1,역삼,Employment,-5534.526659,True,1679,3652,"fare_2025,climate_card"
2,선릉,Employment,-5509.767593,True,1679,3652,"fare_2025,climate_card"
3,신촌,Commercial & Leisure,-5411.782763,True,1679,3652,"fare_2025,climate_card"
4,혜화,Commercial & Leisure,-4407.462213,True,1679,3652,"fare_2025,climate_card"
5,신사,Commercial & Leisure,-5397.113718,True,1679,3652,"fare_2025,climate_card"
6,서울대입구,Residential,-5603.102447,True,1679,3652,"fare_2025,climate_card"
7,수유,Residential,-7131.810639,True,1679,3652,"fare_2025,climate_card"
8,연신내,Residential,-6854.959693,True,1679,3652,"fare_2025,climate_card"


**`eval_event`**: 한 이벤트 × 한 역의 일별 표(`g`)에서 충격·회복을 판정합니다. 충격은 이벤트 기간 중 실제 < 구간 하한인 날이 `min(k, 지속일수)`일 이상, 회복은 해제 후 확인 창(14일, 다음 특보·가림일 전날까지)에서 실제가 구간 안에 `m`일 연속 머문 첫날입니다. 편차 d, 최대 감소율, 누적 손실(이벤트 기간 / 회복 전날까지)도 계산합니다.

In [10]:
ONE = pd.Timedelta(days=1)

def eval_event(g, s, e, k, m):
    """g: 날짜 인덱스의 일별 표(actual, base, lower, upper, mask_day, alert_day). s, e: 이벤트 시작·종료일"""
    imp = g.loc[s:e]
    D = len(imp)
    d = imp.actual / imp.base - 1
    n_below = int((imp.actual < imp.lower).sum())
    out = {'지속일수': D, 'n_below': n_below, '충격': n_below >= min(k, D), '최대감소율%': float(-d.min() * 100), '평균편차%': float(d.mean() * 100),
           '기준선평균': float(imp.base.mean()), '실제평균': float(imp.actual.mean()), '누적손실_기간': float((imp.base - imp.actual).sum())}
    post = g.loc[e + ONE:]
    stop = post.index[post.mask_day | post.alert_day]                        # 다음 특보 또는 가림일이 나오면 그 전날까지만 봄
    truncated, gap = '', np.nan
    if len(stop):
        truncated = '다음특보' if bool(post.loc[stop[0], 'alert_day']) else '가림일'
        gap = (stop[0] - e).days                                              # 해제 후 며칠째에 막혔는가
        post = post.loc[:stop[0] - ONE]
    elif len(post) < EVENT_POST_DAYS:
        truncated = '자료끝'
    inside = ((post.actual >= post.lower) & (post.actual <= post.upper)).astype(int)
    rec = None
    if len(post) >= m:
        run = inside.rolling(m).sum().shift(-(m - 1)) == m                   # t부터 m일이 모두 구간 안인가
        idx = run[run].index
        rec = idx[0] if len(idx) else None
    last = (rec - ONE) if rec is not None else (post.index.max() if len(post) else e)
    out.update({'확인창일수': len(post), '확인창제한': truncated, '막힌날(해제후일차)': gap, '회복일': rec, '회복지연일': (rec - e).days if rec is not None else np.nan, '중도절단': rec is None,
                '누적손실_회복까지': float((g.loc[s:last].base - g.loc[s:last].actual).sum())})
    return out

**placebo 오판정 비율**: 가짜 이벤트(300개 × 9개 역)에 같은 절차를 적용합니다. k는 이 표로 고르고, m은 고정값을 씁니다.

- **충격 오판정(k)**: 가짜 이벤트인데 "구간 아래 날이 k일 이상"으로 충격이라고 판정한 비율. 이 비율이 5% 이하인 가장 작은 k를 고릅니다.
- **회복 오판정(m) — 참고용**: 가짜 이벤트인데 해제 직후부터 구간 안에 m일 연속 머무는 상태로 시작하지 못해(회복지연 > 1일 또는 중도절단) 회복이 늦다고 판정한 비율. **m은 이 표로 고르지 않고 `RECOVERY_M`(=2)로 고정**합니다 (m이 클수록 이 비율은 커집니다).

In [11]:
pl = paths[paths.id.str.startswith('P')]
fk_idx = fk.set_index('id')
rows = []
for (pid, st), g in pl.groupby(['id', '역']):
    g = g.set_index('date')
    s_, e_ = fk_idx.loc[pid, 'start'], fk_idx.loc[pid, 'end']
    for k in KM_CANDIDATES:
        for m in KM_CANDIDATES:
            r = eval_event(g, s_, e_, k, m)
            rows.append({'id': pid, '역': st, 'k': k, 'm': m, '지속일수': r['지속일수'], 'n_below': r['n_below'], '충격': r['충격'], '회복지연일': r['회복지연일'], '중도절단': r['중도절단'], '확인창일수': r['확인창일수']})
plc = pd.DataFrame(rows)
plc['회복늦음'] = plc.중도절단 | (plc.회복지연일 > 1)
st_type = mi.drop_duplicates('station').set_index('station').group
plc['유형'] = plc.역.map(st_type)
plc.to_csv(RESULTS / 'event_placebo.csv', index=False, encoding='utf-8-sig')
rate_k = plc[plc.m == 1].groupby('k').충격.mean()
rate_m = plc[plc.k == 1].groupby('m').회복늦음.mean()
rates = pd.concat([rate_k.rename('충격 오판정 비율(k)'), rate_m.rename('회복 오판정 비율(m)')], axis=1)
rates.index.name = '후보(k 또는 m)'
rates

,충격 오판정 비율(k),회복 오판정 비율(m)
후보(k 또는 m),,
1,0.025185,0.031481
2,0.005556,0.054815
3,0.004815,0.072222


**k 선택과 m 고정**: 위 규칙으로 k를 정하고(m은 `RECOVERY_M`), 유형별 충격 오판정 비율과 구간의 실제 적중률(가짜 이벤트 일수 중 실제가 95% 구간 안인 비율)도 함께 봅니다. 구간이 제대로 보정돼 있으면 적중률이 95% 근처여야 합니다.

In [12]:
ok_k = [k for k in KM_CANDIDATES if rate_k[k] <= FALSE_RATE_MAX]
K = min(ok_k) if ok_k else max(KM_CANDIDATES)
M = RECOVERY_M                                                   # m은 placebo로 고르지 않고 고정
print(f'선택된 k = {K} (placebo 충격 오판정 {rate_k[K]:.1%})' + ('' if ok_k else '  [주의: 5% 이하인 k가 없어 가장 큰 k를 씀]'))
print(f'고정된 m = {M} (참고: placebo 회복 오판정 m=1 {rate_m[1]:.1%}, m=2 {rate_m[2]:.1%}, m=3 {rate_m[3]:.1%})')
inside = ((pl.actual >= pl.lower) & (pl.actual <= pl.upper))
cover = pd.DataFrame({'구간 안': inside.groupby(pl.역.map(st_type)).mean(), '구간 아래': (pl.actual < pl.lower).groupby(pl.역.map(st_type)).mean(), '구간 위': (pl.actual > pl.upper).groupby(pl.역.map(st_type)).mean()})
byk = plc[(plc.k == K) & (plc.m == M)].groupby('유형').충격.mean().rename(f'충격 오판정(k={K})')
pd.concat([cover, byk], axis=1).loc[TYPE_ORDER]

선택된 k = 1 (placebo 충격 오판정 2.5%)
고정된 m = 2 (참고: placebo 회복 오판정 m=1 3.1%, m=2 5.5%, m=3 7.2%)


,구간 안,구간 아래,구간 위,충격 오판정(k=1)
Employment,0.948346,0.029332,0.022322,0.017778
Commercial & Leisure,0.955152,0.026882,0.017967,0.022222
Residential,0.949503,0.037702,0.012794,0.035556


**진짜 이벤트에 적용**: 선택한 k, m으로 모든 분석 이벤트 × 9개 역의 충격·회복을 계산하고, 02번의 이벤트 정보(분류, 종류, 기간, long_event, 민감도 포함 여부, 다음 특보)를 붙입니다. `이력일수`는 시작 전 60일 중 y가 관측된 날 수(기준선의 근거 자료 양)입니다.

In [13]:
meta = ev.set_index('이벤트번호')[['분류', '종류', '시작일', '종료일', '지속일수', '기간', 'long_event', '민감도단일', '다음특보시작일', '다음특보종류']]
real = paths[~paths.id.str.startswith('P')]
rows = []
for (eid, st), g in real.groupby(['id', '역']):
    m_ = meta.loc[eid]
    r = eval_event(g.set_index('date'), m_.시작일, m_.종료일, K, M)
    r.pop('지속일수')
    rows.append({'이벤트번호': eid, '역': st, '유형': st_type[st], **r, '이력일수': int(g.prev_obs60.iloc[0])})
eff = pd.DataFrame(rows).merge(meta.reset_index(), on='이벤트번호')
eff['역'] = pd.Categorical(eff.역, STATIONS, ordered=True)
eff = eff.sort_values(['시작일', '이벤트번호', '역']).reset_index(drop=True)
eff['역'] = eff.역.astype(str)
eff['참고용'] = eff.종류.eq('대설') | (eff.groupby('종류').이벤트번호.transform('nunique') < MIN_EVENTS)
cols = ['이벤트번호', '분류', '종류', '시작일', '종료일', '지속일수', '기간', 'long_event', '민감도단일', '참고용', '역', '유형', 'n_below', '충격', '최대감소율%', '평균편차%', '기준선평균', '실제평균',
        '누적손실_기간', '누적손실_회복까지', '확인창일수', '확인창제한', '막힌날(해제후일차)', '회복일', '회복지연일', '중도절단', '다음특보시작일', '다음특보종류', '이력일수']
eff = eff[cols]
eff.to_csv(RESULTS / 'event_effects.csv', index=False, encoding='utf-8-sig')
real.to_csv(RESULTS / 'event_daily_paths.csv', index=False, encoding='utf-8-sig')
print('event_effects.csv:', eff.shape, '| 이벤트', eff.이벤트번호.nunique(), '개 × 역', eff.역.nunique(), '개')
eff.head(9)

event_effects.csv: (864, 29) | 이벤트 96 개 × 역 9 개


,이벤트번호,분류,종류,시작일,종료일,지속일수,기간,long_event,민감도단일,참고용,...,누적손실_회복까지,확인창일수,확인창제한,막힌날(해제후일차),회복일,회복지연일,중도절단,다음특보시작일,다음특보종류,이력일수
0,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,-2640.600048,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55
1,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,1657.732461,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55
2,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,1509.685594,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55
3,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,-6840.321949,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55
4,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,1327.410779,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55
5,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,4173.638444,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55
6,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,7805.122871,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55
7,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,6421.918158,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55
8,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,False,True,False,...,676.084476,14,,NaN,2014-06-24,1.0,False,NaT,NaN,55


**이벤트 × 역 표 (같은 이벤트에 대한 9개 역의 반응을 나란히)**: 최대 감소율(%)과 회복 지연(일)을 이벤트 줄 × 역 칸으로 펼칩니다. 회복 지연은 충격이 있었던 칸만 값이 있고(중도절단은 `>N`으로 표시, N은 확인한 일수), 충격이 없으면 비웁니다. `results/event_by_station_maxdrop.csv`, `event_by_station_recovery.csv`로 저장합니다.

In [14]:
idx = ['이벤트번호', '분류', '종류', '시작일', '지속일수', '기간']
wide_drop = eff.pivot_table(index=idx, columns='역', values='최대감소율%', observed=True)[STATIONS].round(1)
def rec_text(r):
    if not r.충격: return ''
    return f'>{int(r.확인창일수)}' if r.중도절단 else str(int(r.회복지연일))
eff['회복표시'] = eff.apply(rec_text, axis=1)
wide_rec = eff.pivot(index=idx, columns='역', values='회복표시')[STATIONS]
wide_shock = eff.pivot(index=idx, columns='역', values='충격')[STATIONS].astype(int)
wide_drop.to_csv(RESULTS / 'event_by_station_maxdrop.csv', encoding='utf-8-sig')
wide_rec.to_csv(RESULTS / 'event_by_station_recovery.csv', encoding='utf-8-sig')
wide_drop.head(12)

,,,,,역,시청,역삼,선릉,신촌,혜화,신사,서울대입구,수유,연신내
이벤트번호,분류,종류,시작일,지속일수,기간,,,,,,,,,
단일-대설-03,단일,대설,2015-01-18,1,train,6.1,-0.9,2.1,-1.1,-3.3,0.3,1.6,0.9,0.6
단일-대설-04,단일,대설,2015-12-03,1,train,-3.6,-6.0,-4.0,-3.2,-5.8,-4.6,-2.2,-4.1,-1.3
단일-대설-08,단일,대설,2018-11-24,1,train,7.8,-3.5,-6.0,1.6,0.8,-2.6,2.4,-1.5,3.5
단일-대설-09,단일,대설,2019-02-19,1,train,-0.7,0.9,0.4,1.4,2.9,-0.4,3.2,2.6,3.2
단일-대설-15,단일,대설,2022-12-15,1,train,-5.4,-6.1,-3.2,-1.9,-4.6,-6.1,0.7,2.1,1.7
단일-대설-16,단일,대설,2023-01-06,2,train,21.4,2.2,7.6,5.3,5.4,2.4,3.2,0.1,3.6
단일-대설-17,단일,대설,2023-01-26,1,train,-0.3,-6.2,-0.6,7.3,2.4,-5.6,1.5,0.2,4.0
단일-대설-24,단일,대설,2024-11-27,2,validation,-1.6,2.8,-0.8,-2.0,-0.9,-1.2,3.4,0.3,0.3
단일-폭염-01,단일,폭염,2014-07-09,3,train,1.6,1.5,0.2,0.7,2.0,-1.2,1.4,0.2,0.4


**결과 요약 함수 `summarize`**: 유형 × 종류별로 이벤트 수, 충격 발생 비율, 최대 감소율(평균·중앙값), **회복 판정이 가능했던 건수·이벤트 수(충격이 있고 중도절단이 아닌 경우)**, 회복 지연(평균·중앙값), 중도절단 비율(충격이 있는 경우), 누적 손실 평균을 계산합니다. 대설과 이벤트가 10개 미만인 종류는 `참고용`으로 표시합니다.

In [15]:
def summarize(df):
    rows = []
    for (typ, kind), g in df.groupby(['유형', '종류']):
        sh = g[g.충격]
        rec = sh[~sh.중도절단]
        rows.append({'유형': typ, '종류': kind, '참고용': '참고용' if g.참고용.any() else '', '이벤트 수': g.이벤트번호.nunique(), '이벤트×역': len(g),
                     '충격 비율': round(g.충격.mean(), 3), '최대감소율(평균%)': round(g['최대감소율%'].mean(), 1), '최대감소율(중앙%)': round(g['최대감소율%'].median(), 1),
                     '충격 건수': len(sh), '회복 판정 가능 건수(이벤트×역, 중도절단 제외)': len(rec), '회복 판정 가능 이벤트 수(중도절단 제외)': rec.이벤트번호.nunique(),
                     '회복지연(평균일)': round(rec.회복지연일.mean(), 1) if len(rec) else np.nan, '회복지연(중앙일)': rec.회복지연일.median() if len(rec) else np.nan,
                     '중도절단 비율(충격 중)': round(sh.중도절단.mean(), 3) if len(sh) else np.nan, '누적손실(평균, 명)': round(g.누적손실_회복까지.mean())})
    t = pd.DataFrame(rows)
    t['유형'] = pd.Categorical(t.유형, TYPE_ORDER, ordered=True)
    return t.sort_values(['종류', '유형']).reset_index(drop=True)

### 결과 1: 단일 이벤트 (주 분석)

이벤트 기간 중 다른 종류 특보가 없는 깨끗한 이벤트입니다 (해제 후 7일 안에 다른 특보가 시작된 이벤트 포함, 그 경우 회복은 다음 특보 전날까지만 봄).

In [16]:
main = eff[eff.분류 == '단일']
t_main = summarize(main)
t_main

,유형,종류,참고용,이벤트 수,이벤트×역,충격 비율,최대감소율(평균%),최대감소율(중앙%),충격 건수,"회복 판정 가능 건수(이벤트×역, 중도절단 제외)",회복 판정 가능 이벤트 수(중도절단 제외),회복지연(평균일),회복지연(중앙일),중도절단 비율(충격 중),"누적손실(평균, 명)"
0,Employment,대설,참고용,8,24,0.042,0.1,-0.8,1,1,1,1.0,1.0,0.000,-3001
1,Commercial & Leisure,대설,참고용,8,24,0.000,-0.6,-1.0,0,0,0,NaN,NaN,NaN,-2644
2,Residential,대설,참고용,8,24,0.000,1.2,1.5,0,0,0,NaN,NaN,NaN,641
3,Employment,폭염,,27,81,0.148,5.6,2.9,12,5,3,1.2,1.0,0.583,1565
4,Commercial & Leisure,폭염,,27,81,0.074,5.0,4.9,6,4,2,1.2,1.0,0.333,7050
5,Residential,폭염,,27,81,0.136,3.9,3.2,11,2,1,5.0,5.0,0.818,9260
6,Employment,한파,,19,57,0.053,3.2,1.4,3,3,3,1.0,1.0,0.000,-1488
7,Commercial & Leisure,한파,,19,57,0.123,4.3,2.2,7,6,3,1.0,1.0,0.143,5118
8,Residential,한파,,19,57,0.246,5.1,3.5,14,11,6,1.0,1.0,0.214,8290
9,Employment,호우,,39,117,0.128,4.8,3.5,15,6,3,1.0,1.0,0.600,3680


### 결과 2: 복합 이벤트

다른 종류 특보가 겹친 이벤트를 합쳐서 본 것입니다. 깨끗한 복합 이벤트가 아주 적어 전부 참고용입니다.

In [17]:
comp = eff[eff.분류 == '복합']
t_comp = summarize(comp)
t_comp

,유형,종류,참고용,이벤트 수,이벤트×역,충격 비율,최대감소율(평균%),최대감소율(중앙%),충격 건수,"회복 판정 가능 건수(이벤트×역, 중도절단 제외)",회복 판정 가능 이벤트 수(중도절단 제외),회복지연(평균일),회복지연(중앙일),중도절단 비율(충격 중),"누적손실(평균, 명)"
0,Employment,폭염+호우,참고용,3,9,0.111,5.0,3.1,1,0,0,NaN,NaN,1.0,2539
1,Commercial & Leisure,폭염+호우,참고용,3,9,0.111,4.5,5.6,1,0,0,NaN,NaN,1.0,14095
2,Residential,폭염+호우,참고용,3,9,0.000,3.6,3.0,0,0,0,NaN,NaN,NaN,11271


### 결과 3: 민감도 (종료 후 7일 안에도 다른 종류 특보가 없는 단일 이벤트)

주 분석보다 엄격한 기준입니다. 결과가 크게 달라지지 않는지 확인합니다.

In [18]:
sens = eff[eff.민감도단일]
t_sens = summarize(sens)
t_sens

,유형,종류,참고용,이벤트 수,이벤트×역,충격 비율,최대감소율(평균%),최대감소율(중앙%),충격 건수,"회복 판정 가능 건수(이벤트×역, 중도절단 제외)",회복 판정 가능 이벤트 수(중도절단 제외),회복지연(평균일),회복지연(중앙일),중도절단 비율(충격 중),"누적손실(평균, 명)"
0,Employment,대설,참고용,6,18,0.056,1.3,-0.2,1,1,1,1.0,1.0,0.000,-2017
1,Commercial & Leisure,대설,참고용,6,18,0.000,-0.3,-0.6,0,0,0,NaN,NaN,NaN,-2717
2,Residential,대설,참고용,6,18,0.000,1.1,1.2,0,0,0,NaN,NaN,NaN,348
3,Employment,폭염,,10,30,0.133,4.4,1.2,4,4,2,1.2,1.0,0.000,2822
4,Commercial & Leisure,폭염,,10,30,0.067,3.8,3.0,2,2,1,1.5,1.5,0.000,1339
5,Residential,폭염,,10,30,0.067,2.4,1.2,2,2,1,5.0,5.0,0.000,4839
6,Employment,한파,,17,51,0.059,3.3,1.4,3,3,3,1.0,1.0,0.000,-657
7,Commercial & Leisure,한파,,17,51,0.137,4.6,3.3,7,6,3,1.0,1.0,0.143,6092
8,Residential,한파,,17,51,0.255,5.1,3.4,13,10,5,1.0,1.0,0.231,8459
9,Employment,호우,,17,51,0.137,3.8,3.2,7,6,3,1.0,1.0,0.143,2985


**세 가지 결과를 한 파일로 저장**: `results/event_summary_by_type.csv` (`결과` 열에 단일 / 복합 / 민감도 표시).

In [19]:
summary_all = pd.concat([t_main.assign(결과='단일(주 분석)'), t_comp.assign(결과='복합'), t_sens.assign(결과='민감도')], ignore_index=True)
summary_all = summary_all[['결과'] + [c for c in summary_all.columns if c != '결과']]
summary_all.to_csv(RESULTS / 'event_summary_by_type.csv', index=False, encoding='utf-8-sig')
summary_all.shape

(27, 16)

**m 민감도**: 회복 판정의 연속 일수 m을 1, 2, 3으로 바꾸면 단일(주 분석)의 회복 결과가 어떻게 달라지는지 봅니다. **주 분석은 m=2**입니다. m이 클수록 "구간 안에 오래 머물러야" 회복으로 보므로 지연이 길어지고 중도절단이 늘어납니다. 모든 m에서 회복 판정이 가능했던 건수·이벤트 수(중도절단 제외)를 같이 표시합니다.

In [20]:
rows = []
meta_main = meta[meta.분류 == '단일']
for (eid, st), g in real[real.id.isin(meta_main.index)].groupby(['id', '역']):
    m_ = meta.loc[eid]
    for mm in KM_CANDIDATES:
        r = eval_event(g.set_index('date'), m_.시작일, m_.종료일, K, mm)
        if r['충격']:
            rows.append({'m': mm, '종류': m_.종류, '이벤트': eid, '지연': r['회복지연일'], '중도절단': r['중도절단']})
sm = pd.DataFrame(rows)
ok_rec = sm[~sm.중도절단]
sens_m = pd.DataFrame({'충격 건수': sm.groupby(['종류', 'm']).size(), '회복 판정 가능 건수(중도절단 제외)': ok_rec.groupby(['종류', 'm']).size(),
                       '회복 판정 가능 이벤트 수': ok_rec.groupby(['종류', 'm']).이벤트.nunique(), '회복지연(평균일)': ok_rec.groupby(['종류', 'm']).지연.mean().round(1),
                       '회복지연(중앙일)': ok_rec.groupby(['종류', 'm']).지연.median(), '중도절단 비율': sm.groupby(['종류', 'm']).중도절단.mean().round(3)}).fillna(0)
sens_m.insert(0, '주 분석', ['◀ 주 분석(m=2)' if m == RECOVERY_M else '' for m in sens_m.index.get_level_values('m')])
sens_m.to_csv(RESULTS / 'event_recovery_m_sensitivity.csv', encoding='utf-8-sig')
sens_m

주 분석  충격 건수  회복 판정 가능 건수(중도절단 제외)  회복 판정 가능 이벤트 수  회복지연(평균일)  \
종류 m                                                                        
대설 1                   1                     1               1        1.0   
   2  ◀ 주 분석(m=2)      1                     1               1        1.0   
   3                   1                     1               1        1.0   
폭염 1                  29                    12               5        1.8   
   2  ◀ 주 분석(m=2)     29                    11               4        1.9   
   3                  29                     8               2        2.2   
한파 1                  24                    24               7        1.0   
   2  ◀ 주 분석(m=2)     24                    20               6        1.0   
   3                  24                    20               6        1.0   
호우 1                 100                    81              22        1.0   
   2  ◀ 주 분석(m=2)    100                    45              14        1.0   
   3                 100                    31              10        1.1   

      회복지연(중앙일)  중도절단 비율  
종류 m                      
대설 1        1.0    0.000  
   2        1.0    0.000  
   3        1.0    0.000  
폭염 1        1.0    0.586  
   2        1.0    0.621  
   3        1.5    0.724  
한파 1        1.0    0.000  
   2        1.0    0.167  
   3        1.0    0.167  
호우 1        1.0    0.190  
   2        1.0    0.550  
   3        1.0    0.690

**종류별 합계 (역 유형 구분 없이)**: 단일(주 분석) 결과를 종류별로 합쳐서 봅니다. 민감도 결과와 나란히 놓아 차이를 확인합니다.

In [21]:
def by_kind(df):
    sh = df[df.충격]; rec = sh[~sh.중도절단]
    return pd.DataFrame({'이벤트 수': df.groupby('종류').이벤트번호.nunique(), '충격 비율': df.groupby('종류').충격.mean().round(3),
                         '최대감소율(평균%)': df.groupby('종류')['최대감소율%'].mean().round(1), '충격 건수': sh.groupby('종류').size(),
                         '회복 판정 가능 건수': rec.groupby('종류').size(), '회복 판정 가능 이벤트 수': rec.groupby('종류').이벤트번호.nunique(),
                         '회복지연(평균일)': rec.groupby('종류').회복지연일.mean().round(1), '회복지연(중앙일)': rec.groupby('종류').회복지연일.median(),
                         '중도절단 비율(충격 중)': sh.groupby('종류').중도절단.mean().round(3)}).fillna(0)
pd.concat({'단일(주 분석)': by_kind(main), '민감도': by_kind(sens)}, axis=1)

단일(주 분석)                                                               \
      이벤트 수  충격 비율 최대감소율(평균%) 충격 건수 회복 판정 가능 건수 회복 판정 가능 이벤트 수 회복지연(평균일)   
종류                                                                         
대설        8  0.014        0.3     1           1              1       1.0   
폭염       27  0.119        4.8    29          11              4       1.9   
한파       19  0.140        4.2    24          20              6       1.0   
호우       39  0.285        7.0   100          45             14       1.0   

                             민감도                                      \
   회복지연(중앙일) 중도절단 비율(충격 중) 이벤트 수  충격 비율 최대감소율(평균%) 충격 건수 회복 판정 가능 건수   
종류                                                                     
대설       1.0         0.000     6  0.019        0.7     1           1   
폭염       1.0         0.621    10  0.089        3.5     8           8   
한파       1.0         0.167    17  0.150        4.4    23          19   
호우       1.0         0.550    17  0.275        6.7    42          35   

                                                     
   회복 판정 가능 이벤트 수 회복지연(평균일) 회복지연(중앙일) 중도절단 비율(충격 중)  
종류                                                   
대설              1       1.0       1.0         0.000  
폭염              2       2.2       1.5         0.000  
한파              5       1.0       1.0         0.174  
호우             10       1.0       1.0         0.167

**long_event (지속일수 14일 초과)**: 분석 대상(깨끗한 이벤트) 중 14일을 넘는 이벤트를 따로 봅니다. 긴 이벤트는 대부분 겹침 사유(공휴일 등)로 깨끗한 이벤트에서 빠지므로, 02번의 전체 long_event 현황도 함께 보여 줍니다.

In [22]:
lg = eff[eff.long_event]
print('분석 대상(깨끗한 이벤트) 중 long_event:', lg.이벤트번호.nunique(), '개')
lg_all = ev_all[ev_all.long_event][['이벤트번호', '분류', '종류', '시작일', '종료일', '지속일수', '기간', '깨끗함', '제외사유']]
lg_all.to_csv(RESULTS / 'event_long_events.csv', index=False, encoding='utf-8-sig')
lg_all.reset_index(drop=True)

분석 대상(깨끗한 이벤트) 중 long_event: 0 개


,이벤트번호,분류,종류,시작일,종료일,지속일수,기간,깨끗함,제외사유
0,단일-폭염-08,단일,폭염,2016-07-31,2016-08-25,26,train,False,공휴일·명절·근로자의날·파업
1,단일-폭염-15,단일,폭염,2018-07-12,2018-08-18,38,train,False,공휴일·명절·근로자의날·파업
2,복합-폭염+호우-04,복합,폭염+호우,2021-07-10,2021-08-13,35,train,False,"코로나·메르스, 공휴일·명절·근로자의날·파업"
3,복합-폭염+호우-06,복합,폭염+호우,2023-07-25,2023-08-09,16,train,False,공휴일·명절·근로자의날·파업
4,복합-폭염+호우-10,복합,폭염+호우,2024-07-24,2024-08-31,39,validation,False,공휴일·명절·근로자의날·파업
5,단일-폭염-41,단일,폭염,2025-07-21,2025-08-06,17,test,False,계단더미±30일


**기준선 품질 점검**: ① 학습 구간의 y 관측일수(가림 후), ② 이벤트 근거 자료량(`이력일수`)이 적은 이벤트(60일 중 20일 미만)가 있는지 확인합니다. 근거가 적으면 기준선 구간이 넓어져 충격이 잘 안 잡힙니다.

In [23]:
print('이력일수가 20일 미만인 이벤트:', eff[eff.이력일수 < 20].이벤트번호.nunique(), '개 /', eff.이벤트번호.nunique(), '개')
print('확인창이 제한된 (이벤트×역):', eff.확인창제한.value_counts().to_dict())
nx = eff[eff.확인창제한 == '다음특보']
print('다음 특보로 막힌 경우: 해제 후 7일 안', int((nx['막힌날(해제후일차)'] <= 7).sum()), '건 / 8~14일째', int((nx['막힌날(해제후일차)'] > 7).sum()), '건')
eff.drop_duplicates('이벤트번호').groupby('분류').이력일수.describe().round(1)

이력일수가 20일 미만인 이벤트: 19 개 / 96 개
확인창이 제한된 (이벤트×역): {'다음특보': 657, '': 150, '가림일': 57}
다음 특보로 막힌 경우: 해제 후 7일 안 549 건 / 8~14일째 108 건


,count,mean,std,min,25%,50%,75%,max
분류,,,,,,,,
단일,93.0,33.4,16.4,0.0,22.0,36.0,46.0,60.0
복합,3.0,40.0,1.0,39.0,39.5,40.0,40.5,41.0
